# 08a - Learning curve and choice of the number of folds

Chooses the number k of folds of the cross-validated hyperparameter search (notebook 08). With k folds every model is
trained on (k - 1) / k of the records, so a small k is cheaper but trains on fewer records and gives a pessimistic
validation loss. How much fewer records matter is measured here:

1. the configuration of the `model` and `training` sections is trained on fold 1 of the split (the current validation
   set) with growing shares of its training events (`learning_curve.training_shares`), and the run on the whole
   training set is repeated with another seed;
2. the power law `L(n) = a + b n^(-c)` is fitted to the best validation losses against the training records `n`;
3. for each candidate k (`learning_curve.fold_candidates`) the fitted loss at the training size of k folds is compared
   with the loss on all the records: the increase is the price of k folds;
4. the suggested k is the smallest one whose increase is within the tolerance: `learning_curve.loss_tolerance`, or, if
   null, the difference between the two runs on the whole training set (the variability between runs).

The runs use `training.representation`, `training.feature_source`, the window placement and the time masking of the
configuration, so the result applies to the same variant as the search. Every run is resumable (checkpoint at every
epoch): after an interruption run the notebook again. Total time: about the sum of the shares plus one, times the
duration of one training (about 7 hours on the full traces with time masking).

## 0. Setup

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from wqu.config import load_config
from wqu.data.splitting import SPLIT_COLUMN, TRAIN, VALIDATION
from wqu.training.experiment import (
    best_epoch_metrics,
    build_partition,
    experiment_name,
    load_experiment_records,
    train_partition,
)
from wqu.training.learning_curve import (
    fit_learning_curve,
    fold_table,
    subsample_events,
    suggest_folds,
    training_share_for_folds,
)
from wqu.training.trainer import select_device
from wqu.visualization import apply_plot_style, plot_learning_curve, save_figure

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
CURVE_POINTS = 200

config = load_config(CONFIG_PATH)
columns = config.columns
training = config.training
settings = config.learning_curve
apply_plot_style()
device = select_device(training.device)
output_dir = settings.output_dir / experiment_name(config)
output_dir.mkdir(parents=True, exist_ok=True)

experiment = load_experiment_records(config)
train_records = experiment.records[experiment.records[SPLIT_COLUMN] == TRAIN]
validation_records = experiment.records[experiment.records[SPLIT_COLUMN] == VALIDATION]
print(f"Variant: {experiment_name(config)}; {len(train_records):,} training / {len(validation_records):,} validation "
      f"records ({len(experiment.excluded):,} without data excluded); device {device}")
print(f"Output: {output_dir}")
pd.Series({k: training_share_for_folds(k, config.split.validation_fraction) for k in settings.fold_candidates},
          name="share of the current training set").rename_axis("folds").round(3).to_frame()

## 1. Runs

One run per training share with seed `training.random_seed`, plus the run on the whole training set with seed
`training.random_seed + learning_curve.repeat_seed_offset`. The training events of a smaller share are a subset of
those of a larger one; the normalization statistics and the bad-class weight come from the training records of each
run. Finished runs are not repeated.

In [ ]:
run_specifications = [(share, training.random_seed) for share in sorted(settings.training_shares)]
run_specifications.append((1.0, training.random_seed + settings.repeat_seed_offset))

rows = []
for share, seed in run_specifications:
    run_dir = output_dir / f"share_{share:.3f}_seed_{seed}"
    subset = subsample_events(train_records, share, columns.event_id, settings.subsample_seed)
    partition = build_partition(experiment, training.representation, subset, validation_records, columns.label,
                                run_dir / "normalization.npz")
    print(f"Share {share:.3f}, seed {seed}: {len(subset):,} training records, bad rate "
          f"{subset[columns.label].mean():.3f}")
    result = train_partition(experiment, partition, config.model, replace(training, random_seed=seed),
                             columns.label, run_dir, device)
    rows.append({"share": share, "seed": seed, "training_records": len(subset), **best_epoch_metrics(result)})

runs = pd.DataFrame(rows)
runs.to_csv(output_dir / "runs.csv", index=False)
runs.round(4)

## 2. Learning curve and number of folds

Power law fitted to all the runs; predicted loss and loss increase of each candidate k compared with training on all
the records (training plus validation); suggested k with the tolerance used.

In [ ]:
curve = fit_learning_curve(runs["training_records"].to_numpy(), runs["validation_loss"].to_numpy())
n_available = len(experiment.records)
folds = fold_table(curve, settings.fold_candidates, n_available)

full_runs = runs[runs["share"] == 1.0]["validation_loss"]
seed_difference = float(full_runs.max() - full_runs.min())
tolerance = settings.loss_tolerance if settings.loss_tolerance is not None else seed_difference
suggested = suggest_folds(folds, tolerance)

print(f"Power law: L(n) = {curve.asymptote:.4f} + {curve.amplitude:.4f} (n / 1e4)^(-{curve.exponent:.3f})")
residuals = runs["validation_loss"] - curve.loss(runs["training_records"].to_numpy())
print(f"Largest residual of the fit: {np.abs(residuals).max():.4f}")
print(f"Difference between the two runs on the whole training set: {seed_difference:.4f}")
print(f"Tolerance: {tolerance:.4f} ({'configured' if settings.loss_tolerance is not None else 'from the two seeds'})")
print(f"Suggested number of folds: {suggested}")
folds.assign(within_tolerance=folds["loss_increase"] <= tolerance).to_csv(output_dir / "folds.csv")
folds.assign(within_tolerance=folds["loss_increase"] <= tolerance).round(4)

In [ ]:
sizes = np.linspace(0.8 * runs["training_records"].min(), 1.05 * n_available, CURVE_POINTS)
fig, ax = plt.subplots(figsize=(8, 5))
plot_learning_curve(runs, sizes, curve.loss(sizes), folds, ax)
ax.set_title(f"Learning curve ({experiment_name(config)})")
fig.tight_layout()
save_figure(fig, output_dir / "learning_curve.png")
plt.show()